In [45]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx

DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['day_of_year'] = timestamp_pl.dt.day_of_year
df['weekday'] = timestamp_pl.dt.day_name('Pl')
df['weekday_number'] = timestamp_pl.dt.weekday
df['date'] = timestamp_pl.dt.date
df['localized_time_pl'] = df['timestamp_pl'].dt.tz_localize(None)

In [46]:
dfAK = df[df.osoba == 'AK']
dfAK.sample(3)

,timestamp,app,status,duration,category,osoba,device,timestamp_pl,day_of_year,weekday,weekday_number,date,localized_time_pl
59957,2025-12-20 14:01:14.155000+00:00,MATLAB.exe,not-afk,32.318,studia,AK,PC,2025-12-20 15:01:14.155000+01:00,354,Sobota,5,2025-12-20,2025-12-20 15:01:14.155
62920,2025-12-23 17:00:31.977000+00:00,javaw.exe,not-afk,2530.075,gry komputerowe,AK,PC,2025-12-23 18:00:31.977000+01:00,357,Wtorek,1,2025-12-23,2025-12-23 18:00:31.977
63144,2025-12-23 19:08:45.703000+00:00,aces.exe,not-afk,7.100,gry komputerowe,AK,PC,2025-12-23 20:08:45.703000+01:00,357,Wtorek,1,2025-12-23,2025-12-23 20:08:45.703


Znajdźmy czasy pobudki z każdego dnia

In [47]:
poranki = dfAK[(dfAK['localized_time_pl'].dt.hour < 16) & (dfAK['localized_time_pl'].dt.hour > 5)]

In [48]:
pobudki = poranki.groupby('day_of_year').agg(End=('timestamp_pl','min'))
pobudki.sample(3)

,End
day_of_year,
20,2026-01-20 06:14:01.107000+01:00
24,2026-01-24 07:30:00.424000+01:00
5,2026-01-05 08:30:01.206000+01:00


Znajdźmy czasy pójście spać

In [49]:
wieczory = dfAK[(dfAK['localized_time_pl'].dt.hour > 16) | (dfAK['localized_time_pl'].dt.hour < 5)]
wieczory.sample(3)

,timestamp,app,status,duration,category,osoba,device,timestamp_pl,day_of_year,weekday,weekday_number,date,localized_time_pl
64101,2025-12-30 18:41:33.705000+00:00,chrome.exe,afk,245.203,przeglądarka,AK,PC,2025-12-30 19:41:33.705000+01:00,364,Wtorek,1,2025-12-30,2025-12-30 19:41:33.705
83966,2026-01-24 19:48:51.619000+00:00,Code.exe,not-afk,2.022,oglądanie filmów,AK,PC,2026-01-24 20:48:51.619000+01:00,24,Sobota,5,2026-01-24,2026-01-24 20:48:51.619
76418,2026-01-14 23:23:25.973000+00:00,osk.exe,not-afk,1.020,inne,AK,PC,2026-01-15 00:23:25.973000+01:00,15,Czwartek,3,2026-01-15,2026-01-15 00:23:25.973


Grupuje po dniach, przy czym godziny przed północy wliczam do następnego dnia

In [50]:
wieczory.loc[wieczory['localized_time_pl'].dt.hour>16,'day_of_year'] += 1
# Z pierszego stycznia staje się zero, więc trzeba dać na 365
wieczory.loc[wieczory['day_of_year'] == 366,'day_of_year'] = 1

In [51]:
zasypianie = wieczory.groupby('day_of_year').agg(Start=('timestamp_pl','max'))

In [52]:
zasypianie

,Start
day_of_year,
1,2026-01-01 01:17:52.732000+01:00
2,2026-01-02 02:12:40.221000+01:00
3,2026-01-03 00:00:00+01:00
4,2026-01-04 02:00:18.444000+01:00
5,2026-01-05 01:41:18.814000+01:00
6,2026-01-06 00:00:00+01:00
7,2026-01-07 00:00:00+01:00
8,2026-01-08 00:00:00+01:00
9,2026-01-09 00:01:56.456000+01:00


In [53]:
pobudki

,End
day_of_year,
1,2026-01-01 09:56:49.206000+01:00
2,2026-01-02 10:14:24.814000+01:00
3,2026-01-03 10:55:28.275000+01:00
4,2026-01-04 11:30:16.855000+01:00
5,2026-01-05 08:30:01.206000+01:00
6,2026-01-06 08:30:00.519000+01:00
7,2026-01-07 06:30:01.594000+01:00
8,2026-01-08 08:27:04.512000+01:00
9,2026-01-09 06:30:00.509000+01:00


In [54]:
SenAK = pd.concat([zasypianie,pobudki],axis=1).reset_index()

In [55]:
SenAK['Date'] = SenAK['End'].dt.strftime('%d.%m.%Y')
SenAK['StartTime'] = SenAK['Start'].dt.strftime('%H:%M')
SenAK['EndTime'] = SenAK['End'].dt.strftime('%H:%M')
SenAK.sample(3)

,day_of_year,Start,End,Date,StartTime,EndTime
2,3,2026-01-03 00:00:00+01:00,2026-01-03 10:55:28.275000+01:00,03.01.2026,00:00,10:55
11,12,2026-01-12 00:39:09.439000+01:00,2026-01-12 09:14:59.647000+01:00,12.01.2026,00:39,09:14
15,16,2026-01-16 00:55:11.598000+01:00,2026-01-16 08:35:25.675000+01:00,16.01.2026,00:55,08:35


In [64]:
MojSen = SenAK[['Date','StartTime','EndTime']]
MojSen.head(24).tail(13)

,Date,StartTime,EndTime
11,12.01.2026,00:39,09:14
12,13.01.2026,23:59,06:35
13,14.01.2026,01:03,06:06
14,15.01.2026,01:00,07:42
15,16.01.2026,00:55,08:35
16,17.01.2026,04:44,08:35
17,18.01.2026,01:18,08:35
18,19.01.2026,00:23,07:57
19,20.01.2026,23:37,06:14
20,21.01.2026,23:44,06:30


In [ ]:
#MojSen.head(24).tail(13).to_csv('SenAK2.csv', index=False)

In [58]:
df = df[df['osoba']=='AK']
MostFrequentApps = df.loc[df.category == 'inne',['app','duration']].groupby('app').sum().sort_values('duration',ascending=False) / 3600
MostFrequentApps.head(25)

,duration
app,
QuickPic,2.791135
Zdjęcia,1.364209
Aparat,0.351769
Mapy,0.314071
Cyfrowa równowaga,0.253781
Photos.exe,0.229680
ActivityWatch,0.220423
mBank,0.194146
Trust Wallet,0.051680
